# 📝 과제 02: 온라인 쇼핑 상담 어댑터 평가, 풀 파인튜닝 비교와 병합

## 만들 결과

**교안 02(어댑터 관리와 DoRA)** 의 과제입니다. 과제 01의 온라인 쇼핑 상담 어댑터 4개를 채점해 LoRA Tuned를 고르고, 풀 파인튜닝·DoRA와 비교한 뒤 병합합니다.

**이 과제에서 배우는 것**
1. 어댑터 여러 개 얹기(`load_adapter`)와 바꿔 켜며 평가하기(`set_adapter`)
2. LoRA Tuned와 지난 시간 풀 파인튜닝 비교
3. DoRA(`use_dora=True`) 어댑터와 LoRA Tuned 비교
4. `merge_and_unload`로 병합하고 답과 저장 크기 확인

**만들 것**
- `lora_model`, `model_metrics`(Base(지난 시간), 어댑터 4개, 뒤에 DoRA), `forgetting_table`(Base(지난 시간)와 어댑터 4개)
- `full_compare_table`(풀 파인튜닝 비교), `setting_table`(DoRA 비교)
- `output/commerce_lora_merged`, `same_answer_count`, `size_table`(병합)

## 준비: 데이터와 제공 코드

- **먼저 할 일:** 과제 01을 끝까지 실행합니다. `output/commerce_lora_lr2e-4`에 `checkpoint-<갱신 횟수>` 네 개(갱신 횟수 34, 68, 102, 136)와 MLflow run `lora_lr_2e-4`가 있어야 합니다.
- **API 키:** 판사 Jev(`jev-1.13.0`, `TYPESAFE_MODEL`로 바꿀 수 있습니다)는 `.env`의 `TYPESAFE_API_KEY`가 필요합니다.
- **파일:** 지난 시간 과제 02와 같은 `data/과제/commerce_*.jsonl`, `data/본실습/general_*` 파일을 씁니다. DoRA 학습에는 `data/과제/commerce_val.jsonl`도 씁니다.
- **제공 코드:** 준비, 지난 시간과 같은 생성·판사·지표 함수, 지난 시간 과제 02 결과, LoRA Tuned 고르기, DoRA 학습
- **작성:** 요구사항 번호와 같은 `TODO (번호)` 아래의 `...`를 채우고, 바로 아래 `[자가채점]` 셀을 실행합니다. 앞 문항의 변수를 쓰므로 차례로 풉니다.
- **자가채점:** 판사 점수는 작은 차이로 갈리므로 값이 아니라 테이블의 모양과 계산 관계만 확인합니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
# 지난 시간 과제 데이터의 시스템 프롬프트입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
# 학습 전 모델(Base). 과제 01과 같은 float32 이고, 과제 01의 어댑터를 이 모델 위에 얹습니다
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
print("Base:", MODEL_NAME)

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("commerce-lora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
# 학습, 검증 데이터와 평가에 쓰는 네 가지 자료를 읽습니다
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
val_rows = load_jsonl(data_dir / "과제" / "commerce_val.jsonl")
eval_rows = load_jsonl(data_dir / "과제" / "commerce_eval.jsonl")                       # 쇼핑 상담 평가셋: question_type 은 재현, 적용
out_of_scope_rows = load_jsonl(data_dir / "과제" / "commerce_out_of_scope_eval.jsonl")  # 범위 밖 질문
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")              # 일반 질문과 정답 답변(교안 02와 같은 파일)
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))  # 일반 문단(perplexity)

print("학습:", len(train_rows), "| 검증:", len(val_rows), "| 평가셋:", len(eval_rows), "| 범위 밖:", len(out_of_scope_rows),
      "| 일반 질문:", len(general_rows), "| 일반 문단:", len(general_texts))
print("평가셋 종류별:", pd.DataFrame(eval_rows)["question_type"].value_counts().to_dict())

In [ ]:
# 학습 데이터 표본: 거절 답 행을 뺀 학습 행에서 같은 간격으로 고른 40행(지난 시간 과제 02와 같은 행)
# 거절 답("답할 수 없"이 든 답)은 범위 밖 질문 지표가 따로 보므로 표본에서 뺍니다
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)

# 남은 행 수를 40으로 나눈 간격마다 한 행씩, 앞에서 끝까지 고르게 뽑습니다
# [::sample_gap] 은 sample_gap 칸마다 하나, [:40] 는 그 가운데 앞 40개
sample_gap = len(answer_rows) // 40
train_sample_rows = []
for row in answer_rows[::sample_gap][:40]:
    # 학습한 답(answer)을 정답 답변으로 씁니다
    train_sample_rows.append({"question": row["question"], "reference_answer": row["answer"]})
print("거절 답을 뺀 행 수:", len(answer_rows), "| 뽑는 간격:", sample_gap, "| 표본 수:", len(train_sample_rows))

아래 셀은 지난 시간 과제 02와 같은 판사, 채점 함수, 지표 함수입니다. 실행만 하고 넘어갑니다.

In [ ]:
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

In [ ]:
# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    Noul("actual_output은 input의 질문에 직접 답한다."),
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변을 봅니다
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 쇼핑 상담 질문용: 근거 조문(context)도 보고, 정답 답변에 없는 말이 조문과 어긋나지 않는지 한 문장 더 묻습니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다."""
    # rows 는 모두 같은 종류입니다. 근거 조문(law_text)이 있으면 근거 조문 기준으로 채점합니다
    metric = general_metric
    if "law_text" in rows[0]:
        metric = law_metric

    # 문항마다 테스트 케이스 하나
    test_cases = []
    for index, row in enumerate(rows):
        context = None
        if "law_text" in row:
            context = [row["law_text"]]
        test_cases.append(LLMTestCase(
            name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
            input=row["question"],                    # 질문
            actual_output=answers[index],             # 채점할 모델의 답
            expected_output=row["reference_answer"],  # 정답 답변
            context=context,                          # 근거 조문 원문(없으면 None)
        ))

    # 판사를 동시에 부릅니다. 동시에 보낼 문항 수 상한은 64이고, 반 전체가 같은 키를 쓰다 429(요청 한도 초과)가 나면 줄입니다
    result = evaluate(test_cases=test_cases, metrics=[metric], async_config=AsyncConfig(max_concurrent=64),
                      display_config=DisplayConfig(print_results=False, show_indicator=False))  # 긴 기본 출력을 끕니다

    # 결과를 번호(name)로 찾아 문항 순서대로 놓습니다. 점수는 모든 문장이 참이면 1, 아니면 0이고, 이유에는 문장마다 참일 확률이 나옵니다
    judged = {}
    for test_result in result.test_results:
        judged[test_result.name] = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
    judged_rows = []
    for index in range(len(rows)):
        metric_data = judged[str(index)]
        judged_rows.append({"점수": metric_data.score, "통과": metric_data.success, "이유": metric_data.reason})
    return pd.DataFrame(judged_rows)

In [ ]:
def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 생성하고 채점해, 질문마다 한 행씩 모은 테이블을 돌려줍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    # 거절은 거절 키워드가 답에 있는지로 셉니다(거절률과 과잉 거절률에 씁니다)
    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": "답할 수 없" in answer})
    result_table = pd.DataFrame(result_rows)

    # 정답 답변이 있는 행만 판사가 채점합니다(범위 밖 질문은 정답 답변이 없습니다)
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table

In [ ]:
import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 평균 손실에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            loss = model(input_ids, labels=input_ids).loss.item()  # 입력을 정답으로 주면 다음 토큰 예측 손실의 평균을 돌려줍니다

        # 첫 토큰은 예측하지 않으므로 예측한 토큰 수는 길이 - 1입니다. 평균 * 개수 = 이 문단의 손실 합
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted
    return math.exp(total_loss / total_tokens)


# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나를 앞의 평가 네 가지와 perplexity 로 평가해 사전으로 돌려줍니다."""
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT_COMMERCE),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT_COMMERCE),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT_COMMERCE),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다. 정답률, 거절률은 True 를 1로 본 평균입니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }

**지난 시간 과제 02 결과(제공 코드)**: 지난 시간 `tuned_compare_table`의 `Base` 행과 `Tuned 4에폭(step 136)` 행(풀 파인튜닝 Tuned)을 옮긴 강사 실행 값입니다. 자기 결과가 있으면 바꿔 넣어도 됩니다. Base는 다시 채점해도 답이 같으므로 이 값을 `Base(지난 시간)` 행으로 씁니다.

In [ ]:
# 지난 시간 과제 02 tuned_compare_table 의 두 행(강사 실행 값). 키는 METRIC_NAMES 와 같습니다
day70_base_metrics = {"train_acc": 0.0, "domain_recall_acc": 0.0, "domain_apply_acc": 0.0, "over_refusal_rate": 0.0, "refusal_rate": 0.0, "general_acc": 0.033, "general_ppl": 36.43}
day70_full_ft_metrics = {"train_acc": 0.875, "domain_recall_acc": 0.308, "domain_apply_acc": 0.0, "over_refusal_rate": 0.05, "refusal_rate": 0.0, "general_acc": 0.233, "general_ppl": 50.469}

# 모델별 지표 사전(모델 이름 -> 지표 사전). Base 행은 지난 시간 값이고, 어댑터 지표는 에폭별 평가에서 더합니다
model_metrics = {"Base(지난 시간)": day70_base_metrics}

## 1. 어댑터 얹기: load_adapter

**목적:** 에폭마다 저장한 어댑터 4개를 베이스 모델 하나에 얹어, 다시 학습하지 않고 바꿔 가며 평가할 준비를 합니다.

**요구사항**
- (1) `output/commerce_lora_lr2e-4` 안 `checkpoint-*` 폴더 이름의 갱신 횟수를 정수로 바꿔 작은 순으로 정렬합니다.
- (2) 에폭마다 한 행씩 딕셔너리 리스트 **`adapter_rows`** 를 만듭니다. 키는 `"이름"`(`ep1`~`ep4`), `"에폭"`(1부터), `"갱신 횟수"`, `"폴더"`(그 행의 체크포인트 폴더 `Path`, 예: `adapter_dir / "checkpoint-34"`) 순서입니다.
- (3) 첫 행을 `PeftModel.from_pretrained(base_model, 폴더, adapter_name=이름)`으로 얹어 **`lora_model`** 을 만듭니다.
- (4) 나머지 세 행을 `lora_model.load_adapter(폴더, adapter_name=이름)`으로 더 얹습니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 2-2 어댑터 여러 개 얹기 셀과 같은 방법입니다. 폴더 이름의 하이픈 뒤 숫자가 갱신 횟수입니다.
```

</details>

In [ ]:
adapter_dir = output_dir / "commerce_lora_lr2e-4"
adapter_rows = []
# TODO (1): checkpoint-* 폴더의 갱신 횟수를 int 로 바꿔 정렬하세요
...
# TODO (2): 에폭마다 "이름", "에폭", "갱신 횟수", "폴더" 순서의 딕셔너리를 adapter_rows 에 더하세요
...

# 한 행이 어댑터 하나입니다
display(pd.DataFrame(adapter_rows))

In [ ]:
from peft import PeftModel

first_row = adapter_rows[0]
# TODO (3): 첫 어댑터를 PeftModel.from_pretrained 로 얹으세요
lora_model = ...
# TODO (4): 나머지 어댑터를 load_adapter 로 더 얹으세요
for row in adapter_rows[1:]:
    ...

# peft_config: 얹은 어댑터 이름 -> 그 설정. 켜진 어댑터는 처음 얹은 어댑터 하나입니다
print("얹은 어댑터:", list(lora_model.peft_config.keys()), "| 켜진 어댑터:", lora_model.active_adapter)

In [ ]:
# [자가채점] 어댑터 목록의 이름, 갱신 횟수, 얹은 어댑터를 확인합니다
expected_names = ['ep1', 'ep2', 'ep3', 'ep4']

# 행 수와 행마다 키 순서, 에폭 번호, 폴더
assert isinstance(adapter_rows, list) and len(adapter_rows) == 4, f"adapter_rows가 {len(adapter_rows)}행입니다. 에폭마다 한 행입니다."
names = []
steps_by_row = []
for epoch, row in enumerate(adapter_rows, start=1):
    assert list(row) == ["이름", "에폭", "갱신 횟수", "폴더"], f"행의 키가 {list(row)}입니다. 요구사항 (2)의 순서로 담으세요."
    assert row["에폭"] == epoch, f"{epoch}번째 행의 에폭이 {row['에폭']}입니다. 1부터 셉니다."
    assert isinstance(row["폴더"], Path), f"{row['이름']} 행의 폴더가 Path 객체가 아닙니다."
    assert row["폴더"].name == f"checkpoint-{row['갱신 횟수']}", f"{row['이름']} 행의 폴더 {row['폴더'].name}가 갱신 횟수와 맞지 않습니다."
    names.append(row["이름"])
    steps_by_row.append(row["갱신 횟수"])

# 이름과 갱신 횟수의 순서, 얹은 어댑터 이름
assert names == expected_names, f"이름이 {names}입니다. 에폭 순서로 ep1부터 담으세요."
assert steps_by_row == [34, 68, 102, 136], f"갱신 횟수가 {steps_by_row}입니다. int로 바꿔 정렬했는지 확인하세요."
assert isinstance(lora_model, PeftModel), "lora_model이 PeftModel이 아닙니다. 요구사항 (3)을 확인하세요."
assert sorted(lora_model.peft_config.keys()) == sorted(expected_names), f"얹은 어댑터가 {list(lora_model.peft_config.keys())}입니다. 나머지 세 행도 얹으세요."
print("✅ 통과!")

## 2. 에폭별 어댑터 평가: set_adapter

**목적:** 에폭별 어댑터를 같은 지표로 채점해, 쇼핑 상담에 쓸 어댑터를 고를 근거를 만듭니다.

**요구사항**
- (1) `adapter_rows` 순서대로 반복하며 `lora_model.set_adapter(이름, inference_mode=True)`로 그 어댑터를 켭니다.
- (2) 켠 다음 `evaluate_model(lora_model)`의 반환값(채점 테이블)을 **`epoch_tables[이름]`** 에 담습니다.
- (3) 그 채점 테이블의 `table_metrics(...)`(지표 사전)를 **`epoch_metrics[이름]`** 과 준비 셀의 `model_metrics[이름]`에 담습니다. `model_metrics`는 첫 키 `Base(지난 시간)` 뒤에 어댑터 이름이 차례로 붙습니다.
- (4) `model_metrics`로 모델이 행, 지표가 열인 **`forgetting_table`** 을 만들고 열 이름을 `METRIC_NAMES`로 바꿉니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 2-3 에폭별 어댑터 평가 셀과 같은 방법입니다.
지표 사전으로 DataFrame 을 만들면 키가 열이 되므로 뒤집은 뒤 열 이름을 바꿉니다.
```

</details>

deepeval 은 채점할 때마다 안내(WARNING, Pass Rate)를 찍습니다. 오류가 아니며, 결과는 아래 테이블에서 봅니다.

In [ ]:
epoch_tables = {}
epoch_metrics = {}
for row in adapter_rows:
    # TODO (1): 이 행의 어댑터를 켜세요
    ...
    # TODO (2), (3): 평가해 epoch_tables, epoch_metrics, model_metrics 에 담으세요
    ...

# TODO (4): model_metrics 로 지표 테이블을 만드세요
forgetting_table = ...
display(forgetting_table.round(3))

In [ ]:
# [자가채점] 어댑터별 채점 사전과 지표, 테이블의 행 순서, 켜 둔 어댑터를 확인합니다
expected_names = ['ep1', 'ep2', 'ep3', 'ep4']

# 사전의 키 순서, 채점 결과의 모양, 지표가 table_metrics 로 다시 계산한 값과 같은지
assert list(epoch_tables) == expected_names and list(epoch_metrics) == expected_names, "epoch_tables, epoch_metrics의 키는 adapter_rows의 이름 순서입니다."
for name in expected_names:
    assert list(epoch_tables[name]) == ["train", "domain", "refusal", "general", "ppl"], f"epoch_tables[{name}]에 evaluate_model의 반환값을 담으세요."
    recomputed = table_metrics(epoch_tables[name])
    for metric_name in METRIC_NAMES:
        assert abs(epoch_metrics[name][metric_name] - recomputed[metric_name]) < 1e-9, f"epoch_metrics[{name}]이 table_metrics(epoch_tables[{name}])와 다릅니다."
    assert model_metrics[name] == epoch_metrics[name], f"model_metrics[{name}]에도 같은 지표를 담으세요."

# 테이블의 행, 열 순서와 반복이 끝난 뒤 켜져 있는 어댑터
assert list(model_metrics) == ["Base(지난 시간)"] + expected_names, f"model_metrics의 키가 {list(model_metrics)}입니다. Base(지난 시간) 뒤에 어댑터 이름이 와야 합니다."
assert list(forgetting_table.index) == ["Base(지난 시간)"] + expected_names, "forgetting_table을 반복이 끝난 model_metrics로 만드세요."
assert list(forgetting_table.columns) == list(METRIC_NAMES.values()), "forgetting_table의 열 이름을 METRIC_NAMES로 바꾸세요."
assert lora_model.active_adapter == expected_names[-1], f"켜진 어댑터가 {lora_model.active_adapter}입니다. 평가 전에 그 행의 어댑터를 켜세요."
# 켜져 있는 마지막 어댑터로 perplexity 를 다시 계산해 그 이름의 평가값과 비교합니다
last_ppl = epoch_tables[expected_names[-1]]["ppl"]
assert abs(perplexity(lora_model, general_texts) - last_ppl) < last_ppl * 1e-4, "마지막 어댑터의 perplexity가 다시 계산한 값과 다릅니다. set_adapter 뒤에 evaluate_model을 불렀는지 확인하세요."

# 어댑터마다 도메인 답이 달라야 반복마다 어댑터를 바꿔 켠 것입니다
domain_answer_sets = set()
for name in expected_names:
    domain_answer_sets.add(tuple(epoch_tables[name]["domain"]["답변"]))
assert len(domain_answer_sets) > 1, "어댑터 4개의 도메인 답이 모두 같습니다. 반복마다 어댑터를 바꿔 켰는지 확인하세요."
print("✅ 통과!")

## LoRA Tuned 고르기(제공 코드)

교안 02와 같은 규칙(지난 시간 규칙)으로 고릅니다. 학습 데이터 정답률 0.8 이상인 후보 가운데 쇼핑 상담 평가셋 통과 수가 가장 많은 것, 후보가 없으면 학습 데이터 정답률이 가장 높은 것입니다(같으면 앞 에폭). 실행만 하고 넘어갑니다.

- **결과:** `tuned_row`, `tuned_name`, `tuned_step`과 행 이름표 `tuned_label`(뒤의 모든 테이블에 씁니다)

In [ ]:
TRAIN_ACC_MIN = 0.8   # 규칙 1: 학습 데이터 정답률이 이 값 이상인 어댑터만 후보

# 규칙 2: 후보 가운데 쇼핑 상담 평가셋 통과 수가 가장 많은 것(앞 에폭부터 보므로 같으면 앞 에폭)
tuned_row = None
for row in adapter_rows:
    passes = epoch_tables[row["이름"]]["domain"]["통과"].sum()   # 재현 + 적용 통과 문항 수
    if epoch_metrics[row["이름"]]["train_acc"] >= TRAIN_ACC_MIN:
        if tuned_row is None or passes > best_passes:
            tuned_row = row
            best_passes = passes

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 것(같으면 앞 에폭)
if tuned_row is None:
    tuned_row = adapter_rows[0]
    for row in adapter_rows:
        if epoch_metrics[row["이름"]]["train_acc"] > epoch_metrics[tuned_row["이름"]]["train_acc"]:
            tuned_row = row

# 고른 어댑터의 이름과, 테이블에 쓸 이름표(지난 시간 Tuned 이름표와 같은 모양)
tuned_name = tuned_row["이름"]
tuned_epoch = tuned_row["에폭"]
tuned_step = tuned_row["갱신 횟수"]
tuned_label = f"LoRA Tuned {tuned_epoch}에폭(step {tuned_step})"
print("고른 어댑터:", tuned_label, "| 어댑터 이름:", tuned_name, "| 폴더:", tuned_row["폴더"])

## 3. 지난 시간 풀 파인튜닝과 비교

**목적:** LoRA Tuned와 지난 시간 풀 파인튜닝을 같은 지표의 한 테이블로 비교합니다. 지난 시간 풀 파인튜닝은 낮은 학습률(lr 5e-6)로 학습했으므로 방법만의 차이는 아닙니다.

**요구사항**
- (1) 세 사전을 행으로 둔 **`full_compare_table`** 을 만듭니다. 행 이름과 값은 차례로 `"Base(지난 시간)"`: `day70_base_metrics`, `tuned_label`: `epoch_metrics[tuned_name]`, `"풀 파인튜닝 Tuned(지난 시간)"`: `day70_full_ft_metrics`입니다.
- (2) 열 이름을 `METRIC_NAMES`로 바꿉니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 3절 풀 파인튜닝 비교 셀과 같은 방법입니다. 행 이름을 키로, 지표 사전을 값으로 둔 딕셔너리로 테이블을 만들고 뒤집습니다.
```

</details>

In [ ]:
# TODO (1), (2): 세 사전을 행으로 두고 열 이름을 METRIC_NAMES 로 바꾸세요
full_compare_table = ...
display(full_compare_table.round(3))

In [ ]:
# [자가채점] 비교 테이블의 행, 열, 값을 확인합니다
# 행 순서와 열 이름
expected_rows = ["Base(지난 시간)", tuned_label, "풀 파인튜닝 Tuned(지난 시간)"]
assert list(full_compare_table.index) == expected_rows, f"full_compare_table의 행이 {list(full_compare_table.index)}입니다. 요구사항 (1)의 순서로 두세요."
assert list(full_compare_table.columns) == list(METRIC_NAMES.values()), "full_compare_table의 열 이름을 METRIC_NAMES로 바꾸세요."

# 행마다 값이 원래 사전의 값과 같은지
for name, column in METRIC_NAMES.items():
    assert abs(full_compare_table.loc["Base(지난 시간)", column] - day70_base_metrics[name]) < 1e-9, "\"Base(지난 시간)\" 행은 day70_base_metrics입니다."
    assert abs(full_compare_table.loc[tuned_label, column] - epoch_metrics[tuned_name][name]) < 1e-9, "tuned_label 행은 epoch_metrics[tuned_name]입니다."
    assert abs(full_compare_table.loc["풀 파인튜닝 Tuned(지난 시간)", column] - day70_full_ft_metrics[name]) < 1e-9, "\"풀 파인튜닝 Tuned(지난 시간)\" 행은 day70_full_ft_metrics입니다."
print("✅ 통과!")

## 4. DoRA 비교: use_dora

**목적:** DoRA와 LoRA Tuned를 학습 파라미터 수, 학습 시간, 같은 지표의 한 테이블로 비교합니다.

**요구사항**
- (1) LoRA Tuned와 같은 설정(`r=8`, `lora_alpha=16`, `target_modules="all-linear"`, `lora_dropout=0.05`, `task_type="CAUSAL_LM"`)에 `use_dora=True`, `exclude_modules=["layers.0.linear_attn.in_proj_qkv"]`를 더한 **`dora_config`** 를 만듭니다. 뺀 층은 값이 모두 0인 행이 있어 DoRA 손실이 NaN이 되는 층입니다.
- (2) 제공 코드 셀을 실행해 과제 01과 같은 일정으로 DoRA를 학습합니다. 검증 손실이 NaN이면 그 셀에서 멈춥니다.
- (3) 고른 에폭의 DoRA 체크포인트 `dora_dir`을 `lora_model.load_adapter(dora_dir, adapter_name="dora")`로 얹고 `lora_model.set_adapter("dora", inference_mode=True)`로 켭니다.
- (4) `evaluate_model`과 `table_metrics`로 평가한 지표를 `model_metrics["DoRA"]`에 담습니다.
- (5) 다음 제공 코드 셀이 학습 파라미터 수, 학습 시간(초), 지표를 한 테이블 **`setting_table`** 로 만듭니다. 학습 시간은 같은 GPU 종류에서 잰 값끼리만 비교합니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 4-2 DoRA 학습, 4-3 DoRA 평가 셀과 같은 방법입니다. LoRA Tuned 설정에 인자 두 개만 더합니다.
어댑터를 얹고 켜는 것은 교안 02 2-2, 2-3과 같습니다.
```

</details>

In [ ]:
from peft import LoraConfig

# TODO (1): use_dora=True, exclude_modules 를 더한 LoraConfig 를 만드세요
# 나머지 다섯 값은 과제 01의 lora_config 와 같습니다
dora_config = ...

In [ ]:
# [자가채점] DoRA 설정을 확인합니다(학습 전에 확인합니다)
# DoRA 켜기와 뺀 층(peft 가 리스트를 집합으로 바꿔 둡니다), 나머지는 LoRA Tuned 와 같은 값
assert isinstance(dora_config, LoraConfig) and dora_config.use_dora is True, "dora_config에 use_dora=True를 넣으세요."
assert dora_config.exclude_modules == {"layers.0.linear_attn.in_proj_qkv"}, f"exclude_modules가 {dora_config.exclude_modules}입니다. 요구사항 (1)의 층 하나만 리스트로 넣으세요."
assert dora_config.r == 8 and dora_config.lora_alpha == 16 and dora_config.lora_dropout == 0.05, "r, lora_alpha, lora_dropout은 LoRA Tuned와 같은 값입니다."
assert dora_config.target_modules == "all-linear" and dora_config.task_type == "CAUSAL_LM", "target_modules, task_type은 LoRA Tuned와 같은 값입니다. DoRA 학습 뒤라면 이 셀부터 다시 실행하세요."
print("✅ 통과!")

In [ ]:
# (2) 제공 코드: 새 베이스 모델에 DoRA 어댑터를 붙이고, 과제 01과 같은 일정으로 학습합니다
import gc

from datasets import Dataset
from peft import get_peft_model
from trl import SFTConfig, SFTTrainer

# 과제 01과 같은 학습, 검증 Dataset
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])

# lora_model 의 베이스에는 이미 어댑터 4개가 얹혀 있으므로 DoRA 는 새 베이스 모델에 붙입니다
dora_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
# 층마다 A, B 와 크기 벡터 m 을 붙이고, 그 셋만 학습하도록 베이스 가중치를 고정합니다
dora_model = get_peft_model(dora_base, dora_config)

# 과제 01의 sft_config 와 같은 값에 output_dir, run_name 만 바꿉니다
dora_args = SFTConfig(
    output_dir=str(output_dir / "commerce_dora"),
    learning_rate=2e-4,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    logging_steps=10,
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="dora",
)

# 과제 01과 같은 학습기 구성. 에폭마다 commerce_dora/checkpoint-<갱신 횟수> 에 어댑터를 저장합니다
dora_trainer = SFTTrainer(model=dora_model, args=dora_args, train_dataset=train_dataset, eval_dataset=val_dataset,
                          processing_class=tokenizer)
with mlflow.start_run(run_name=dora_args.run_name):
    # 교안 02와 같은 네 항목. get_peft_model 이 target_modules 를 실제 층 이름으로 바꿔 두므로 대상 층은 지정한 값을 적습니다
    mlflow.log_params({"lora_r": dora_config.r, "lora_alpha": dora_config.lora_alpha, "target_modules": "all-linear",
                       "use_dora": dora_config.use_dora})
    dora_output = dora_trainer.train()

# 검증 손실이 NaN 이면 DoRA 의 크기 나누기가 0/0 이 된 것이므로, 판사 채점 전에 여기서 멈춥니다
# log_history 에서 검증 손실만 모아 마지막 에폭 값(맨 끝)이 유한한 수인지 봅니다
dora_eval_losses = []
for log in dora_trainer.state.log_history:
    if "eval_loss" in log:
        dora_eval_losses.append(log["eval_loss"])
assert math.isfinite(dora_eval_losses[-1]), f"DoRA 검증 손실이 {dora_eval_losses[-1]}입니다. dora_config의 exclude_modules를 확인하세요."

# 학습기와 모델을 지워 GPU 메모리를 비웁니다(어댑터는 에폭마다 저장했습니다)
# gc.collect: 지운 객체의 메모리를 바로 회수합니다, empty_cache: PyTorch 가 잡아 둔 빈 GPU 메모리를 돌려줍니다
del dora_trainer, dora_model, dora_base
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# LoRA Tuned 와 같은 갱신 횟수의 DoRA 체크포인트를 골라 같은 학습량끼리 비교합니다
dora_dir = output_dir / "commerce_dora" / f"checkpoint-{tuned_step}"
# TODO (3): DoRA 어댑터를 "dora" 이름으로 얹고 켜세요
...
# TODO (4): 평가한 지표를 담으세요
model_metrics["DoRA"] = ...

In [ ]:
# (5) 제공 코드: LoRA Tuned 와 DoRA 의 학습 파라미터 수, 학습 시간, 지표 일곱 개를 한 테이블로 봅니다
from safetensors.torch import load_file

# LoRA Tuned 의 학습 시간: 과제 01 run 에 학습기가 남긴 train_runtime(초). 같은 이름이 여러 개면 가장 최근 것(맨 위)
# search_runs 는 run 하나를 한 행으로 돌려주고, 열 이름은 tags.mlflow.runName, metrics.train_runtime 처럼 붙습니다
runs_table =mlflow.search_runs(experiment_names=["commerce-lora"], order_by=["start_time DESC"])
tuned_run = runs_table[runs_table["tags.mlflow.runName"] == "lora_lr_2e-4"].iloc[0]
tuned_seconds = round(tuned_run["metrics.train_runtime"])

# 학습 파라미터 수: 어댑터 파일에 저장된 텐서의 원소 수를 더합니다(어댑터 파일에는 학습 파라미터만 저장됩니다)
tuned_count = 0
for tensor in load_file(tuned_row["폴더"] / "adapter_model.safetensors").values():
    tuned_count += tensor.numel()
dora_count = 0
for tensor in load_file(dora_dir / "adapter_model.safetensors").values():
    dora_count += tensor.numel()

# 설정마다 학습 파라미터 수와 학습 시간 한 행, 그 옆에 지표 열을 행 이름(설정)으로 이어 붙입니다(join)
setting_info = pd.DataFrame([
    {"설정": tuned_label, "학습 파라미터 수": tuned_count, "학습 시간(초)": tuned_seconds},
    {"설정": "DoRA", "학습 파라미터 수": dora_count, "학습 시간(초)": round(dora_output.metrics["train_runtime"])},
]).set_index("설정")
setting_table = setting_info.join(pd.DataFrame({tuned_label: epoch_metrics[tuned_name], "DoRA": model_metrics["DoRA"]}).T.rename(columns=METRIC_NAMES))
display(setting_table.round(3))

In [ ]:
# [자가채점] 끝까지 돈 학습, 얹은 어댑터, 비교 테이블을 확인합니다
# NaN 없이 끝까지 돈 학습, "dora" 이름으로 얹어 켠 어댑터
assert math.isfinite(model_metrics["DoRA"]["general_ppl"]), "DoRA의 general_ppl이 NaN입니다. exclude_modules를 넣었는지 확인하세요."
assert dora_output.global_step == 136, f"DoRA 학습의 global_step이 {dora_output.global_step}입니다(기대값 136)."
assert "dora" in lora_model.peft_config and lora_model.peft_config["dora"].use_dora is True, "DoRA 어댑터를 \"dora\" 이름으로 얹으세요."
assert lora_model.active_adapter == "dora", "얹은 뒤 \"dora\" 어댑터를 켜세요."

# 켜져 있는 DoRA 어댑터로 perplexity 를 다시 계산해 담은 지표와 비교합니다
dora_ppl = model_metrics["DoRA"]["general_ppl"]
tuned_ppl = epoch_metrics[tuned_name]["general_ppl"]
assert abs(perplexity(lora_model, general_texts) - dora_ppl) < dora_ppl * 1e-4, "다시 계산한 perplexity가 model_metrics[\"DoRA\"]와 다릅니다. 켠 뒤에 평가했는지 확인하세요."
assert abs(dora_ppl - tuned_ppl) > tuned_ppl * 1e-4, "DoRA 지표가 LoRA Tuned와 같습니다. DoRA 어댑터를 켜고 평가하세요."

# setting_table 의 행, 열과 파라미터 수 차(크기 m 원소 수 - 뺀 층의 A, B 원소 수), 지표 값
assert list(setting_table.index) == [tuned_label, "DoRA"], f"setting_table의 행이 {list(setting_table.index)}입니다."
assert list(setting_table.columns) == ["학습 파라미터 수", "학습 시간(초)"] + list(METRIC_NAMES.values()), f"setting_table의 열이 {list(setting_table.columns)}입니다."
assert dora_count - tuned_count == 336448, f"DoRA와 LoRA Tuned의 학습 파라미터 수 차이가 {dora_count - tuned_count}입니다(기대값 336,448). exclude_modules가 한 층인지 확인하세요."
for name, column in METRIC_NAMES.items():
    assert abs(setting_table.loc["DoRA", column] - model_metrics["DoRA"][name]) < 1e-9, "DoRA 행의 지표는 model_metrics[\"DoRA\"]입니다."
    assert abs(setting_table.loc[tuned_label, column] - epoch_metrics[tuned_name][name]) < 1e-9, "tuned_label 행의 지표는 epoch_metrics[tuned_name]입니다."
print("✅ 통과!")

## 5. 병합과 저장: merge_and_unload

**목적:** 고른 어댑터를 베이스 가중치에 합쳐, 어댑터 없이 바로 배포할 수 있는 모델 하나로 만듭니다.

**요구사항**
- (1) 준비 셀처럼(`dtype=torch.float32`, `.to(device)`) 학습 전 모델을 새로 불러 **`merge_base`** 를 만듭니다. 어댑터가 여럿 얹힌 `lora_model`은 쓰지 않습니다.
- (2) `PeftModel.from_pretrained(merge_base, tuned_row["폴더"])`로 고른 어댑터를 얹고 `.merge_and_unload()`로 병합한 **`merged_model`** 을 만듭니다.
- (3) `generate_replies(merged_model, eval_questions, SYSTEM_PROMPT_COMMERCE)`의 답 리스트를 **`merged_answers`** 에 담습니다.
- (4) `merged_answers`와 고른 어댑터의 답(`epoch_tables[tuned_name]["domain"]["답변"]`)을 차례로 비교해 글자까지 같은 답의 수를 정수 **`same_answer_count`** 로 셉니다. 병합의 반올림 차이로 한두 문항은 다를 수 있습니다.
- (5) `merged_model`과 `tokenizer`를 `save_pretrained`로 `merged_dir`(`output/commerce_lora_merged`)에 저장합니다.
- (6) 열이 `"저장물"`, `"크기(MB)"`인 **`size_table`** 을 만듭니다. `"저장물"` 열 값은 `"고른 어댑터"`, `"병합 모델"` 순서이고, `"크기(MB)"`는 각각 `tuned_row["폴더"] / "adapter_model.safetensors"`, `merged_dir / "model.safetensors"` 파일의 크기(바이트를 `1024 ** 2`로 나눈 MB)입니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 6-1 병합, 6-2 병합 전과 후의 답 비교, 6-3 병합 모델 저장 셀과 같은 방법입니다.
크기 테이블도 교안 02 6-3과 같은 방법으로 파일 크기를 MB 로 바꿉니다.
```

</details>

In [ ]:
# TODO (1): 학습 전 모델을 새로 불러오세요
merge_base = ...
# TODO (2): 고른 어댑터를 얹고 병합하세요
merged_model = ...

# 병합 뒤에는 PeftModel 이 아닌 원래 모델 클래스입니다
print("병합 모델:", type(merged_model).__name__)

In [ ]:
# 평가셋 질문만 순서대로 모읍니다(고른 어댑터의 도메인 채점 결과와 같은 문항 순서)
eval_questions = []
for row in eval_rows:
    eval_questions.append(row["question"])
# TODO (3): 병합 모델로 답을 생성하세요
merged_answers = ...

# zip 으로 같은 문항의 두 답을 짝지어 비교합니다
same_answer_count = 0
for adapter_answer, merged_answer in zip(epoch_tables[tuned_name]["domain"]["답변"], merged_answers):
    # TODO (4): 두 답이 글자까지 같으면 1을 더하세요
    ...
print("같은 답:", same_answer_count, "/", len(merged_answers))

In [ ]:
merged_dir = output_dir / "commerce_lora_merged"
# TODO (5): 병합 모델과 토크나이저를 저장하세요
...
# TODO (6): 두 파일의 크기(MB) 테이블을 만드세요
size_table = ...
display(size_table)

In [ ]:
# [자가채점] 병합 모델의 모양, 답 비교, 저장 크기를 확인합니다
# 병합 모델은 PeftModel 이 아니고 층이 원래 Linear, 답은 평가 문항마다 문자열 하나
assert not isinstance(merged_model, PeftModel), "merged_model에 merge_and_unload()의 반환값을 담으세요."
assert type(merged_model.model.layers[0].mlp.gate_proj) is torch.nn.Linear, f"병합 모델의 gate_proj가 {type(merged_model.model.layers[0].mlp.gate_proj).__name__}입니다. 병합이 안 됐습니다."
assert isinstance(merged_answers, list) and len(merged_answers) == len(eval_rows), "merged_answers는 평가 문항마다 답 하나인 리스트입니다."
for merged_answer in merged_answers:
    assert isinstance(merged_answer, str), "merged_answers에는 generate_replies의 답 문자열을 담습니다."

# 두 답 리스트를 Series 로 놓고 == 로 비교한 True 의 수
adapter_answers = pd.Series(list(epoch_tables[tuned_name]["domain"]["답변"]))
expected_same = int((adapter_answers == pd.Series(merged_answers)).sum())
assert isinstance(same_answer_count, int) and same_answer_count == expected_same, f"same_answer_count가 {same_answer_count}입니다. 같은 답마다 1씩 더했는지 확인하세요."
assert len(merged_answers) - 2 <= same_answer_count <= len(merged_answers), f"같은 답이 {same_answer_count}/{len(merged_answers)}개입니다. 병합한 어댑터가 tuned_row[\"폴더\"]인지 확인하세요."

# 크기 테이블의 모양, 토크나이저 저장, 병합 모델이 어댑터보다 훨씬 큰지
assert list(size_table.columns) == ["저장물", "크기(MB)"] and list(size_table["저장물"]) == ["고른 어댑터", "병합 모델"], "size_table의 열과 행 순서를 요구사항 (6)과 맞추세요."
assert (merged_dir / "tokenizer_config.json").exists(), "merged_dir에 토크나이저가 없습니다. 토크나이저도 저장하세요."
assert size_table["크기(MB)"].iloc[1] > size_table["크기(MB)"].iloc[0] * 10, "병합 모델 크기가 어댑터의 10배 이하입니다. merged_dir의 model.safetensors 크기를 읽었는지 확인하세요."
print("✅ 통과!")